WEEK 5
I USED Sample - Superstore.csv DATASET FOR THIS

Q1: What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?


A1 = 1) Hadoop MapReduce repeatedly reads from and writes to disk, making it slow. Spark keeps data in memory (RAM), reducing expensive disk operations and speeding up processing.
2) Machine learning and graph algorithms require multiple passes over the same data. Spark uses .cache() to store data in memory, avoiding repeated disk reads and making these workloads much faster.
3) MapReduce is limited to separate Map and Reduce stages, often requiring multiple jobs for complex tasks. Spark uses a DAG (Directed Acyclic Graph) execution engine to optimize the entire workflow before execution.

Q2: Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

A2) 1) In Hadoop MapReduce, iterative machine learning algorithms (such as K-Means and Gradient Descent) must repeatedly read the same data from disk in every iteration, causing significant delays due to disk I/O.
2) Subsequent iterations access data directly from memory, eliminating repeated disk reads and reducing execution time. This makes Spark up to 100× faster than traditional disk-based systems for iterative machine learning tasks.
3)Spark allows datasets to be stored in memory using .cache() or .persist(). After the first read, the data remains in RAM instead of being reloaded from disk.

In [51]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Superstore_Spark_Assignment") \
    .config("spark.sql.shuffle.partitions", "5") \
    .getOrCreate()

df = spark.read.csv("Sample - Superstore.csv", header=True, inferSchema=True)

In [52]:
# Q3 (I HAVE PRITNED THE FIRST 20 ROWS)
df_dedup = df.dropDuplicates(["Customer ID", "Order Date"])
print(f"Original Row Count: {df.count()} | Deduplicated Row Count: {df_dedup.count()}")
df_dedup.select("Row ID", "Customer ID", "Order Date", "Sales").show(20)

Original Row Count: 9994 | Deduplicated Row Count: 4992
+------+-----------+----------+--------+
|Row ID|Customer ID|Order Date|   Sales|
+------+-----------+----------+--------+
|  1300|   AA-10315| 10/4/2015|   26.96|
|  5199|   AA-10315|  3/3/2016|3930.072|
|  2230|   AA-10315| 3/31/2014| 673.568|
|  1160|   AA-10315| 6/29/2017|  362.94|
|  7469|   AA-10315| 9/15/2014|   14.94|
|  3008|   AA-10375|10/24/2014|  34.272|
|  6466|   AA-10375|11/13/2015|   84.96|
|  2264|   AA-10375|11/14/2016|  499.98|
|  6748|   AA-10375|12/11/2017|  14.952|
|   808|   AA-10375|  2/3/2015|    28.4|
|  1173|   AA-10375| 4/21/2014|   16.52|
|  1979|   AA-10375|  5/8/2015|   5.248|
|   536|   AA-10375| 7/10/2016|  16.768|
|  9539|   AA-10375|  9/7/2017|    16.9|
|    13|   AA-10480| 4/15/2017|  15.552|
|  1460|   AA-10480|  5/4/2014|   27.46|
|  3108|   AA-10480| 7/17/2016|   21.93|
|  7703|   AA-10480| 8/26/2016|   11.56|
|  8008|   AA-10645|11/17/2015|   Sand"|
|  7490|   AA-10645| 11/5/2017|   12.96|
+

In [53]:
#Q4
from pyspark.sql.functions import avg, col

df_clean_sales = df.withColumn("Sales", col("Sales").try_cast("double"))

df_sales_analysis = df_clean_sales.filter((col("Region") == "West") & col("Sales").isNotNull()) \
    .groupBy("Category") \
    .agg(avg("Sales").alias("average_sale_amount"))

df_sales_analysis.show(20)

+---------------+-------------------+
|       Category|average_sale_amount|
+---------------+-------------------+
|     Technology| 422.64417449664415|
|Office Supplies| 117.48907552370453|
|      Furniture| 360.59540420899896|
+---------------+-------------------+



Q5) .na.drop(): Removes entire rows that contain missing or null values in the specified columns. It is useful when incomplete records should be discarded.
.na.fill(): Keeps all rows but replaces missing or null values with a specified default value such as 0 AND Unknown

In [54]:
# Q5 THERE WAS NO STSTUS COLUMN IN THE DATASET
from pyspark.sql.functions import lit

df_with_status = df.withColumn("status", lit(None).cast("string"))
df_status_cleaned = df_with_status.na.fill({"status": "Unknown"})
df_status_cleaned.select("Row ID", "Order ID", "status").show(20)

+------+--------------+-------+
|Row ID|      Order ID| status|
+------+--------------+-------+
|     1|CA-2016-152156|Unknown|
|     2|CA-2016-152156|Unknown|
|     3|CA-2016-138688|Unknown|
|     4|US-2015-108966|Unknown|
|     5|US-2015-108966|Unknown|
|     6|CA-2014-115812|Unknown|
|     7|CA-2014-115812|Unknown|
|     8|CA-2014-115812|Unknown|
|     9|CA-2014-115812|Unknown|
|    10|CA-2014-115812|Unknown|
|    11|CA-2014-115812|Unknown|
|    12|CA-2014-115812|Unknown|
|    13|CA-2017-114412|Unknown|
|    14|CA-2016-161389|Unknown|
|    15|US-2015-118983|Unknown|
|    16|US-2015-118983|Unknown|
|    17|CA-2014-105893|Unknown|
|    18|CA-2014-167164|Unknown|
|    19|CA-2014-143336|Unknown|
|    20|CA-2014-143336|Unknown|
+------+--------------+-------+
only showing top 20 rows


In [55]:
# Q6
from pyspark.sql.functions import col

df_city_counts = df.groupBy("City") \
    .count() \
    .filter(col("count") > 100) \
    .orderBy("count", ascending=False)

df_city_counts.show(20)

+-------------+-----+
|         City|count|
+-------------+-----+
|New York City|  915|
|  Los Angeles|  747|
| Philadelphia|  537|
|San Francisco|  510|
|      Seattle|  428|
|      Houston|  377|
|      Chicago|  314|
|     Columbus|  222|
|    San Diego|  170|
|  Springfield|  163|
|       Dallas|  157|
| Jacksonville|  125|
|      Detroit|  115|
+-------------+-----+



Q7

* Spark DataFrames are **immutable**, so operations like `.drop()` or `.withColumnRenamed()` create a **new DataFrame** instead of changing the original.
* Spark uses **lazy execution**, meaning changes are only applied when an action like `.show()` or `.save()` is run.
* This is **memory-efficient** because Spark avoids copying the entire dataset.
* It also makes code **cleaner** by allowing multiple operations to be chained together.


In [56]:
# Q7
from pyspark.sql.functions import col

df_cleaned = df.withColumnRenamed("Postal Code", "postal_code") \
               .withColumnRenamed("Product ID", "product_id") \
               .drop("Row ID")

df_cleaned.select("postal_code", "product_id", "Sales").show(20)

+-----------+---------------+--------+
|postal_code|     product_id|   Sales|
+-----------+---------------+--------+
|      42420|FUR-BO-10001798|  261.96|
|      42420|FUR-CH-10000454|  731.94|
|      90036|OFF-LA-10000240|   14.62|
|      33311|FUR-TA-10000577|957.5775|
|      33311|OFF-ST-10000760|  22.368|
|      90032|FUR-FU-10001487|   48.86|
|      90032|OFF-AR-10002833|    7.28|
|      90032|TEC-PH-10002275| 907.152|
|      90032|OFF-BI-10003910|  18.504|
|      90032|OFF-AP-10002892|   114.9|
|      90032|FUR-TA-10001539|1706.184|
|      90032|TEC-PH-10002033| 911.424|
|      28027|OFF-PA-10002365|  15.552|
|      98103|OFF-BI-10003656| 407.976|
|      76106|OFF-AP-10002311|   68.81|
|      76106|OFF-BI-10000756|   2.544|
|      53711|OFF-ST-10004186|  665.88|
|      84084|OFF-ST-10000107|    55.5|
|      94109|OFF-AR-10003056|    8.56|
|      94109|TEC-PH-10001949|  213.48|
+-----------+---------------+--------+
only showing top 20 rows


In [57]:
# Q8
from pyspark.sql.functions import col, lit

df_mocked = df.withColumn("age", lit(25)) \
              .withColumn("subscription", lit("Premium"))

df_filtered = df_mocked.filter(
    (col("age") >= 18) &
    (col("age") <= 30) &
    (col("subscription") == "Premium")
)

df_filtered.select("Row ID", "Order ID", "age", "subscription", "Sales").show(20)

+------+--------------+---+------------+--------+
|Row ID|      Order ID|age|subscription|   Sales|
+------+--------------+---+------------+--------+
|     1|CA-2016-152156| 25|     Premium|  261.96|
|     2|CA-2016-152156| 25|     Premium|  731.94|
|     3|CA-2016-138688| 25|     Premium|   14.62|
|     4|US-2015-108966| 25|     Premium|957.5775|
|     5|US-2015-108966| 25|     Premium|  22.368|
|     6|CA-2014-115812| 25|     Premium|   48.86|
|     7|CA-2014-115812| 25|     Premium|    7.28|
|     8|CA-2014-115812| 25|     Premium| 907.152|
|     9|CA-2014-115812| 25|     Premium|  18.504|
|    10|CA-2014-115812| 25|     Premium|   114.9|
|    11|CA-2014-115812| 25|     Premium|1706.184|
|    12|CA-2014-115812| 25|     Premium| 911.424|
|    13|CA-2017-114412| 25|     Premium|  15.552|
|    14|CA-2016-161389| 25|     Premium| 407.976|
|    15|US-2015-118983| 25|     Premium|   68.81|
|    16|US-2015-118983| 25|     Premium|   2.544|
|    17|CA-2014-105893| 25|     Premium|  665.88|


Q9)
Avoid incorrect results: Too many null values can make averages and other calculations misleading.
Prevent errors: If all values are null, functions like sum() or avg() may return null or NaN.
Keep data consistent: Filling or removing nulls before calculations ensures cleaner, more reliable results in later analysis.

In [58]:
# Q9
from pyspark.sql.functions import avg, col

df_clean_sales = df.withColumn("Sales", col("Sales").try_cast("double"))
df_imputed = df_clean_sales.na.fill({"Sales": 0.0})

df_result = df_imputed.groupBy("Category") \
                      .agg(avg("Sales").alias("average_sales"))

df_result.show(20)

+---------------+------------------+
|       Category|     average_sales|
+---------------+------------------+
|     Technology|452.57177422847667|
|Office Supplies| 116.7445947560576|
|      Furniture|345.61379599245623|
+---------------+------------------+



In [59]:
# Q10
from pyspark.sql.functions import col, to_timestamp

df_with_time = df.withColumn("raw_timestamp", col("Order Date"))

df_final = df_with_time.withColumn("event_time", to_timestamp(col("raw_timestamp"), "M/d/yyyy")) \
                       .drop("raw_timestamp")

df_final.select("Row ID", "Order ID", "event_time").show(20)


+------+--------------+-------------------+
|Row ID|      Order ID|         event_time|
+------+--------------+-------------------+
|     1|CA-2016-152156|2016-11-08 00:00:00|
|     2|CA-2016-152156|2016-11-08 00:00:00|
|     3|CA-2016-138688|2016-06-12 00:00:00|
|     4|US-2015-108966|2015-10-11 00:00:00|
|     5|US-2015-108966|2015-10-11 00:00:00|
|     6|CA-2014-115812|2014-06-09 00:00:00|
|     7|CA-2014-115812|2014-06-09 00:00:00|
|     8|CA-2014-115812|2014-06-09 00:00:00|
|     9|CA-2014-115812|2014-06-09 00:00:00|
|    10|CA-2014-115812|2014-06-09 00:00:00|
|    11|CA-2014-115812|2014-06-09 00:00:00|
|    12|CA-2014-115812|2014-06-09 00:00:00|
|    13|CA-2017-114412|2017-04-15 00:00:00|
|    14|CA-2016-161389|2016-12-05 00:00:00|
|    15|US-2015-118983|2015-11-22 00:00:00|
|    16|US-2015-118983|2015-11-22 00:00:00|
|    17|CA-2014-105893|2014-11-11 00:00:00|
|    18|CA-2014-167164|2014-05-13 00:00:00|
|    19|CA-2014-143336|2014-08-27 00:00:00|
|    20|CA-2014-143336|2014-08-2

Q11
**Shuffle in `groupBy()`**

When you use **`groupBy()`**, Spark moves rows with the same key to the same partition so they can be grouped and aggregated. This data movement across executors is called a **shuffle**.

It is a **wide transformation** because data is exchanged between multiple partitions. Since shuffling involves disk I/O and network transfer, it is one of the most expensive operations in Spark.


In [60]:
# Q 12
from pyspark.sql.functions import col, lit

df_mocked = df.withColumn("email", lit(None).cast("string")) \
              .withColumn("username", lit(""))

df_cleaned = df_mocked.filter(
    col("email").isNotNull() &
    (col("username") != "") &
    col("username").isNotNull()
)

df_cleaned.select("Row ID", "Order ID", "email", "username").show(20)


+------+--------+-----+--------+
|Row ID|Order ID|email|username|
+------+--------+-----+--------+
+------+--------+-----+--------+



In [61]:
# Q13
from pyspark.sql.functions import avg, col, max, min

df_clean = df.withColumn("price", col("Sales").try_cast("double"))

df_stats = df_clean.groupBy("Category") \
    .agg(
        min("price").alias("min_price"),
        max("price").alias("max_price"),
        avg("price").alias("mean_price")
    )

df_stats.show(20)

+---------------+---------+---------+------------------+
|       Category|min_price|max_price|        mean_price|
+---------------+---------+---------+------------------+
|     Technology|     0.99| 22638.48| 454.5405475802047|
|Office Supplies|    0.444|  9892.74|121.69225531914947|
|      Furniture|    1.892| 4416.174| 353.4459311957568|
+---------------+---------+---------+------------------+



Q 14
Wrong data type: If date formats are inconsistent, Spark may treat the entire column as a string instead of a date.
Missing values: Dates that don't match the inferred format can be converted to null without warning.
Analysis issues: Incorrect date types or null values can cause errors in sorting, filtering, and date-based calculations.

In [62]:
#Q15
from pyspark.sql.functions import col, lit, sum

df_mocked = df.withColumn("store_id", lit("Store_A")) \
              .withColumn("price", col("Sales").try_cast("double"))

df_cleaned = df_mocked.dropDuplicates() \
                      .na.fill({"price": 0.0})

df_pipeline = df_cleaned.groupBy("store_id") \
                        .agg(sum("price").alias("total_revenue"))

df_pipeline.show(20)

+--------+------------------+
|store_id|     total_revenue|
+--------+------------------+
| Store_A|2272449.8562999573|
+--------+------------------+

